# 🩸 HCV Data Mining — Klasifikasi & Clustering
**Dataset:** HCV (Hepatitis C Virus) — UCI Machine Learning Repository  
**Metode:** Random Forest Classification (Binary) + K-Means Clustering  
**Target:** Prediksi status pasien: Donor Sehat (0) vs Indikasi Penyakit Hati (1)

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, f1_score,
                              ConfusionMatrixDisplay)
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from imblearn.over_sampling import SMOTE
import joblib

print('Libraries loaded successfully.')

## 2. Load & Pemahaman Data (Data Understanding)

In [ ]:
df = pd.read_csv('hcvdat0.csv')
df = df.drop(columns=['Unnamed: 0'])  # drop index kolom

print('Shape:', df.shape)
print()
df.head()

In [ ]:
print('=== Info Dataset ===')
df.info()
print()
print('=== Statistik Deskriptif ===')
df.describe().round(2)

In [ ]:
print('=== Distribusi Target (Category) ===')
print(df['Category'].value_counts())
print()
print('=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# Visualisasi distribusi kategori
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original 5 class
counts = df['Category'].value_counts()
colors = ['#2ecc71','#27ae60','#e74c3c','#c0392b','#8e44ad']
axes[0].bar(counts.index, counts.values, color=colors[:len(counts)], edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribusi Kategori Original (5 Kelas)', fontweight='bold')
axes[0].set_xlabel('Kategori')
axes[0].set_ylabel('Jumlah')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(counts.values):
    axes[0].text(i, v+3, str(v), ha='center', fontweight='bold')

# Binary target
binary_counts = pd.Series([540, 75], index=['Donor Sehat (0)', 'Indikasi Penyakit Hati (1)'])
axes[1].bar(binary_counts.index, binary_counts.values, color=['#2ecc71','#e74c3c'], edgecolor='white', linewidth=1.5)
axes[1].set_title('Distribusi Target Binary', fontweight='bold')
axes[1].set_ylabel('Jumlah')
for i, v in enumerate(binary_counts.values):
    axes[1].text(i, v+3, f'{v} ({v/615*100:.1f}%)', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('dist_kategori.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Preprocessing Data

In [ ]:
# 3.1 Buat target binary
df['target'] = df['Category'].apply(lambda x: 0 if '0' in str(x) else 1)
print('Target distribution:')
print(df['target'].value_counts())

# 3.2 Encode Sex
df['Sex_enc'] = (df['Sex'] == 'm').astype(int)

# 3.3 Definisi fitur
FEATURES = ['Age', 'Sex_enc', 'ALB', 'ALP', 'ALT', 'AST', 'BIL', 'CHE', 'CHOL', 'CREA', 'GGT', 'PROT']
TARGET = 'target'

X = df[FEATURES]
y = df[TARGET]

print(f'\nFitur: {FEATURES}')
print(f'Jumlah fitur: {len(FEATURES)}')

In [ ]:
# 3.4 Imputasi missing values dengan median
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=FEATURES)

print('Missing values setelah imputasi:')
print(X_imputed.isnull().sum().sum(), '(seharusnya 0)')

In [ ]:
# 3.5 Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train positive rate: {y_train.mean():.3f}')
print(f'Test positive rate: {y_test.mean():.3f}')

In [ ]:
# 3.6 SMOTE untuk mengatasi class imbalance (hanya pada train set)
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('Sebelum SMOTE:', y_train.value_counts().to_dict())
print('Setelah SMOTE:', pd.Series(y_train_sm).value_counts().to_dict())

# 3.7 Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sm)
X_test_scaled = scaler.transform(X_test)

## 4. EDA — Exploratory Data Analysis

In [ ]:
# Heatmap korelasi
fig, ax = plt.subplots(figsize=(12, 9))
corr = X_imputed.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            mask=mask, ax=ax, linewidths=0.5, square=True)
ax.set_title('Heatmap Korelasi Antar Fitur', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('heatmap_korelasi.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribusi biomarker per kelas
df_plot = X_imputed.copy()
df_plot['target'] = y.values

bio_features = ['ALT', 'AST', 'GGT', 'BIL', 'ALB', 'CHE']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(bio_features):
    for label, color, name in [(0,'#2ecc71','Sehat'), (1,'#e74c3c','Penyakit Hati')]:
        data = df_plot[df_plot['target']==label][feat]
        axes[i].hist(data, bins=30, alpha=0.6, color=color, label=name, edgecolor='white')
    axes[i].set_title(f'Distribusi {feat}', fontweight='bold')
    axes[i].legend()
    axes[i].spines[['top','right']].set_visible(False)

plt.suptitle('Distribusi Biomarker: Sehat vs Penyakit Hati', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('distribusi_biomarker.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Metode 1 — Classification (Random Forest)

In [ ]:
# 5.1 Baseline models comparison
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
}

print('=== Perbandingan Model (5-Fold CV F1 Macro) ===')
cv_results = {}
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train_sm,
                                 cv=5, scoring='f1_macro')
    cv_results[name] = cv_scores
    print(f'{name:25s}: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
# 5.2 Train final Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train_sm)
print('Model trained.')

In [ ]:
# 5.3 Evaluasi pada test set
y_pred = rf_model.predict(X_test_scaled)
y_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Donor Sehat', 'Penyakit Hati']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}')
print(f'F1-Score Macro: {f1_score(y_test, y_pred, average="macro"):.4f}')

In [ ]:
# 5.4 Confusion Matrix + ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Donor Sehat', 'Penyakit Hati'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix', fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'Random Forest (AUC = {auc_score:.3f})')
axes[1].plot([0,1],[0,1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('confusion_roc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5.5 Feature Importance
importances = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
colors_fi = ['#e74c3c' if v >= importances.quantile(0.75) else '#3498db' for v in importances.values]
importances.plot(kind='barh', ax=ax, color=colors_fi, edgecolor='white')
ax.set_title('Feature Importance — Random Forest', fontweight='bold')
ax.set_xlabel('Importance Score')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 fitur paling berpengaruh:')
print(importances.sort_values(ascending=False).head())

## 6. Metode 2 — Clustering (K-Means)

In [ ]:
# 6.1 Elbow Method + Silhouette Score
X_cluster = scaler.transform(X_imputed)  # gunakan semua data

inertia = []
silhouette = []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(X_cluster, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(K_range, inertia, 'bo-', linewidth=2)
axes[0].set_title('Elbow Method', fontweight='bold')
axes[0].set_xlabel('Jumlah Cluster (k)')
axes[0].set_ylabel('Inertia')
axes[0].spines[['top','right']].set_visible(False)

axes[1].plot(K_range, silhouette, 'rs-', linewidth=2)
axes[1].set_title('Silhouette Score per k', fontweight='bold')
axes[1].set_xlabel('Jumlah Cluster (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = K_range[np.argmax(silhouette)]
print(f'\nBest k by Silhouette Score: {best_k} (score={max(silhouette):.4f})')

In [ ]:
# 6.2 Final K-Means
BEST_K = 3  # adjust based on elbow/silhouette
kmeans = KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster)

df_cluster = X_imputed.copy()
df_cluster['cluster'] = cluster_labels
df_cluster['target'] = y.values

sil = silhouette_score(X_cluster, cluster_labels)
print(f'K-Means (k={BEST_K}) Silhouette Score: {sil:.4f}')

print('\nDistribusi target per cluster:')
print(df_cluster.groupby('cluster')['target'].value_counts().unstack(fill_value=0))

In [ ]:
# 6.3 PCA Visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_cluster)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cluster_colors = ['#3498db','#e74c3c','#2ecc71','#f39c12']

# Plot by cluster
for cl in range(BEST_K):
    mask = cluster_labels == cl
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=cluster_colors[cl], label=f'Cluster {cl}', alpha=0.7, s=40)
axes[0].set_title('Cluster K-Means (PCA 2D)', fontweight='bold')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# Plot by actual label
for label, color, name in [(0,'#2ecc71','Donor Sehat'),(1,'#e74c3c','Penyakit Hati')]:
    mask = y.values == label
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=color, label=name, alpha=0.7, s=40)
axes[1].set_title('Label Aktual (PCA 2D)', fontweight='bold')
axes[1].legend()
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('pca_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Variance explained by 2 PC: {pca.explained_variance_ratio_.sum():.3f}')

In [ ]:
# 6.4 Profil tiap cluster
cluster_profile = df_cluster.groupby('cluster')[['ALT','AST','GGT','BIL','ALB','CHE','CHOL','CREA']].mean().round(2)
print('=== Profil Rata-Rata Biomarker per Cluster ===')
print(cluster_profile)

## 7. Simpan Model & Artefak

In [ ]:
joblib.dump(rf_model, 'model_hcv.pkl')
joblib.dump(scaler, 'scaler_hcv.pkl')
joblib.dump(imputer, 'imputer_hcv.pkl')
joblib.dump(kmeans, 'kmeans_hcv.pkl')
joblib.dump(pca, 'pca_hcv.pkl')

# Simpan juga data cluster untuk visualisasi app
df_cluster_export = X_imputed.copy()
df_cluster_export['cluster'] = cluster_labels
df_cluster_export['target'] = y.values
df_cluster_export['pca1'] = X_pca[:, 0]
df_cluster_export['pca2'] = X_pca[:, 1]
df_cluster_export.to_csv('cluster_data.csv', index=False)

print('Semua model dan artefak berhasil disimpan.')
print('Files: model_hcv.pkl, scaler_hcv.pkl, imputer_hcv.pkl, kmeans_hcv.pkl, pca_hcv.pkl, cluster_data.csv')

In [ ]:
# Ringkasan akhir
print('='*55)
print('           RINGKASAN HASIL DATA MINING')
print('='*55)
print(f'Dataset    : HCV (Hepatitis C Virus) — UCI')
print(f'Records    : 615 | Fitur: 12')
print(f'Target     : Binary (Donor Sehat vs Penyakit Hati)')
print()
print('[ Classification — Random Forest ]')
print(f'  F1-Score Macro : {f1_score(y_test, y_pred, average="macro"):.4f}')
print(f'  ROC-AUC        : {roc_auc_score(y_test, y_proba):.4f}')
print()
print('[ Clustering — K-Means ]')
print(f'  k Optimal      : {BEST_K}')
print(f'  Silhouette     : {sil:.4f}')
print('='*55)